# Introduction
This notebook is used to fetch, plot, and analyze experiment results.

## 1. Initial Setup

In [1]:
print("Hello World")
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

Hello World
Free GPU Memory (GB): 18.4355


In [2]:
print("\n################################")
print("Setting up environment...")
print("################################\n")

import os
#os.chdir('..')
print("Current Working Directory ", os.getcwd())
import sys
sys.path.append("../") # Add directory containing src/data to path

import importlib
import src  # Assuming src is the package name

# Reload the src module after making changes
importlib.reload(src)

%load_ext autoreload
%autoreload 2


################################
Setting up environment...
################################

Current Working Directory  /nfs/homedirs/daro/git/quantization-reliability
Initializing src package
Initializing src package


## 2. Fetch Results

In [4]:
import seml
import pandas as pd

db_collection = 'llama-pert-awq-bnb-hqq'
states = ["COMPLETED"]

# Get the results
all_results = seml.evaluation.get_results(db_collection, to_data_frame=True, states=states)

print(f"Length of all_results before deduplication: {len(all_results)}")

# Get the list of columns that start with 'config.'
config_columns = [col for col in all_results.columns if col.startswith('config.')]

# Drop duplicates based on config columns, keeping the last occurrence
all_results = all_results.drop_duplicates(subset=config_columns, keep='last')

print(f"Length of all_results after deduplication: {len(all_results)}")
print("Columns used for deduplication:")
print(config_columns)
print("\nAll columns in the dataframe:")
print(all_results.columns)
all_results.head()

Output()

Output()

Length of all_results before deduplication: 4800
Length of all_results after deduplication: 4800
Columns used for deduplication:
['config.overwrite', 'config.db_collection', 'config.batch_size', 'config.dataset_name', 'config.device', 'config.exp_id', 'config.max_entries', 'config.max_new_tokens', 'config.model_name', 'config.n_beams', 'config.n_repeats', 'config.num_excel_rows', 'config.save_excel', 'config.seed', 'config.strategy', 'config.temperature', 'config.typo_intensity', 'config.typo_type', 'config.use_beam_search']

All columns in the dataframe:
Index(['_id', 'config.overwrite', 'config.db_collection', 'config.batch_size',
       'config.dataset_name', 'config.device', 'config.exp_id',
       'config.max_entries', 'config.max_new_tokens', 'config.model_name',
       'config.n_beams', 'config.n_repeats', 'config.num_excel_rows',
       'config.save_excel', 'config.seed', 'config.strategy',
       'config.temperature', 'config.typo_intensity', 'config.typo_type',
       'config

,_id,config.overwrite,config.db_collection,config.batch_size,config.dataset_name,config.device,config.exp_id,config.max_entries,config.max_new_tokens,config.model_name,...,result.Brier_adj,result.LogLoss_adj,result.Entropy_adj,result.AUCROC_sem,result.AUCPR_sem,result.Brier_sem,result.LogLoss_sem,result.Entropy_sem,result.Accuracy,result.fail_trace
0,1,1,llama-pert-awq-bnb-hqq,32,P101,cuda,pert-awq-bnb-hqq-10-07,None,25,Llama-3-8B,...,0.561755,5.918077,36.896553,1.0,1.0,0.0,2.220446e-16,-3.981839e-08,0.396552,<function get_results at 0x7f08933dc0d0>
1,2,2,llama-pert-awq-bnb-hqq,32,P101,cuda,pert-awq-bnb-hqq-10-07,None,25,Llama-3-8B,...,0.561755,5.918077,36.896553,1.0,1.0,0.0,2.220446e-16,-3.981839e-08,0.396552,<function get_results at 0x7f08933dc0d0>
2,3,3,llama-pert-awq-bnb-hqq,32,P101,cuda,pert-awq-bnb-hqq-10-07,None,25,Llama-3-8B,...,0.561755,5.918077,36.896553,1.0,1.0,0.0,2.220446e-16,-3.981839e-08,0.396552,<function get_results at 0x7f08933dc0d0>
3,4,4,llama-pert-awq-bnb-hqq,32,P101,cuda,pert-awq-bnb-hqq-10-07,None,25,Llama-3-8B,...,0.579266,5.823508,45.919863,1.0,1.0,0.0,2.220446e-16,-3.765434e-08,0.375000,<function get_results at 0x7f08933dc0d0>
4,5,5,llama-pert-awq-bnb-hqq,32,P101,cuda,pert-awq-bnb-hqq-10-07,None,25,Llama-3-8B,...,0.624608,6.945512,41.225135,1.0,1.0,0.0,2.220446e-16,-3.246064e-08,0.323276,<function get_results at 0x7f08933dc0d0>


### 2.1 Check Failed Rows

In [8]:
import seml
import pandas as pd
from collections import Counter

db_collection = 'llama-pert-awq-bnb-hqq'
states = ["FAILED"]

# Get the results
failed_results = seml.evaluation.get_results(db_collection, to_data_frame=True, states=states)
print(f"Length of failed_results before deduplication: {len(failed_results)}")

# Get the list of columns that start with 'config.'
config_columns = [col for col in failed_results.columns if col.startswith('config.')]

# Drop duplicates based on config columns, keeping the last occurrence
failed_results = failed_results.drop_duplicates(subset=config_columns, keep='last')
print(f"Length of failed_results after deduplication: {len(failed_results)}")

print("\nUnique values and their frequencies for each config parameter in failed_results:")
for col in ["config.dataset_name", "config.model_name", "config.typo_type", "config.typo_intensity"]:
    value_counts = Counter(failed_results[col])
    print(f"\n{col}:")
    for value, count in value_counts.items():
        print(f"  - {value}: {count}")

Output()

Output()

Length of failed_results before deduplication: 70
Length of failed_results after deduplication: 70

Unique values and their frequencies for each config parameter in failed_results:

config.dataset_name:
  - P364: 11
  - P37: 18
  - P740: 21
  - P101: 20

config.model_name:
  - Llama-3-8B-BNB-4bit-local: 50
  - Llama-3-8B-HQQ-mixed-local: 20

config.typo_type:
  - word_phrase_translation: 2
  - word_context_aware_insertion: 1
  - word_remove_punctuation: 5
  - word_keyword_only: 5
  - word_taxonomy_pos: 6
  - word_taxonomy_neg: 5
  - none: 5
  - char_insertion: 5
  - char_deletion: 6
  - char_replacement: 4
  - char_repetition: 5
  - char_swapping: 3
  - word_CMW: 4
  - char_LCC: 5
  - word_synonym: 1
  - char_insert_noise: 3
  - word_repeat: 1
  - char_substitution: 3
  - word_emoji: 1

config.typo_intensity:
  - 1: 24
  - 2: 23
  - 3: 23


### 2.2 Check high accuracy columns

In [9]:
# Filter rows where accuracy is higher than 0.6
high_accuracy_results = all_results[all_results['result.Accuracy'] > 0.6]

# Print the number of rows that meet this criteria
print(f"Number of rows with accuracy > 0.6: {len(high_accuracy_results)}")

# Display the first few rows of the filtered results
print(high_accuracy_results[['config.model_name', 'config.strategy', 'result.Accuracy']].head())

# Count the number of high accuracy rows for each unique model name
model_counts = high_accuracy_results['config.model_name'].value_counts()

print("\nNumber of high accuracy rows for each model:")
print(model_counts)

# Optional: Calculate and print the percentage of high accuracy rows for each model
total_rows = len(all_results)
model_percentages = (model_counts / total_rows * 100).round(2)

print("\nPercentage of high accuracy rows for each model:")
print(model_percentages)

Number of rows with accuracy > 0.6: 2656
   config.model_name    config.strategy  result.Accuracy
54        Llama-3-8B  Direct Completion         0.669540
55        Llama-3-8B  Direct Completion         0.744253
56        Llama-3-8B  Direct Completion         0.778736
60        Llama-3-8B  Direct Completion         0.939611
61        Llama-3-8B  Direct Completion         0.939611

Number of high accuracy rows for each model:
config.model_name
Llama-3-8B                    756
Llama-3-8B-AWQ-4bit-local     714
Llama-3-8B-BNB-4bit-local     620
Llama-3-8B-HQQ-mixed-local    566
Name: count, dtype: int64

Percentage of high accuracy rows for each model:
config.model_name
Llama-3-8B                    15.75
Llama-3-8B-AWQ-4bit-local     14.88
Llama-3-8B-BNB-4bit-local     12.92
Llama-3-8B-HQQ-mixed-local    11.79
Name: count, dtype: float64


## 3. Plot results

## 3.1 Final plots

In [5]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os
from fpdf import FPDF
import numpy as np
from PIL import Image

from src.algorithms.quantization import get_model_num_bits

# Define color scheme for the four models
model_colors = {
    'Llama-3-8B': '#1f77b4',  # Blue
    'Llama-3-8B-AWQ-4bit-local': '#ff7f0e',  # Orange
    'Llama-3-8B-BNB-4bit-local': '#2ca02c',  # Green
    'Llama-3-8B-HQQ-mixed-local': '#d62728'  # Red
}

# Define color scheme for intensities
intensity_colors = {
    1: '#1f77b4',  # Blue
    2: '#2ca02c',  # Green
    3: '#d62728'   # Red
}

def create_comparison_radar_plots(df, metric, intensity):
    fig = make_subplots(rows=1, cols=3, specs=[[{'type': 'polar'}]*3],
                        subplot_titles=["Llama vs AWQ", "Llama vs BNB", "Llama vs HQQ"])
    
    df_intensity = df[df['config.typo_intensity'] == intensity]
    base_model = 'Llama-3-8B'
    comparison_models = ['Llama-3-8B-AWQ-4bit-local', 'Llama-3-8B-BNB-4bit-local', 'Llama-3-8B-HQQ-mixed-local']
    
    for i, comp_model in enumerate(comparison_models):
        for model, color in [(base_model, model_colors[base_model]), (comp_model, model_colors[comp_model])]:
            df_model = df_intensity[df_intensity['config.model_name'] == model]
            
            if df_model.empty:
                print(f"Warning: No data for model {model} with intensity {intensity} and metric {metric}")
                continue
            
            values = df_model[f'result.{metric}'].tolist()
            if not values:
                print(f"Warning: No values for model {model} with intensity {intensity} and metric {metric}")
                continue
            
            values.append(values[0])  # Close the polygon
            
            theta = df_model['config.typo_type'].tolist()
            theta.append(theta[0])  # Close the polygon
            
            num_bits = get_model_num_bits(model)
            
            fig.add_trace(go.Scatterpolar(
                r=values,
                theta=theta,
                fill='toself',
                name=f"{model} ({num_bits}-bit)",
                line=dict(color=color)
            ), row=1, col=i+1)
    
    fig.update_layout(
        height=600, width=1800,
        title=f'Comparison of {metric} for Intensity {intensity}',
        font=dict(size=10),
    )
    
    for i in range(1, 4):
        fig.update_layout(**{
            f'polar{i}': dict(
                radialaxis=dict(visible=True, range=[0, 1]),
                angularaxis=dict(
                    tickfont=dict(size=8),
                    rotation=90,
                    direction="clockwise"
                )
            )
        })
    
    return fig

def create_boxplot_comparison(df, metric):
    fig = go.Figure()
    
    for model in model_colors.keys():
        num_bits = get_model_num_bits(model)
        for intensity in [1, 2, 3]:
            df_subset = df[(df['config.model_name'] == model) & (df['config.typo_intensity'] == intensity)]
            fig.add_trace(go.Box(
                y=df_subset[f'result.{metric}'],
                x=df_subset['config.typo_intensity'],
                name=f'{model} ({num_bits}-bit, Intensity {intensity})',
                marker_color=model_colors[model],
                showlegend=intensity == 1
            ))
    
    fig.update_layout(
        title=f'Distribution of {metric} across Intensities and Models',
        xaxis_title='Intensity',
        yaxis_title=metric,
        boxmode='group',
        height=600,
        width=1200,
    )
    return fig

def create_all_perturbations_bar_plot(df, metric, model):
    fig = go.Figure()
    
    baseline_value = df[(df['config.typo_type'] == 'none') & (df['config.model_name'] == model)][f'result.{metric}'].iloc[0]
    perturbation_types = df['config.typo_type'].unique()
    perturbation_types = [p for p in perturbation_types if p != 'none']
    
    for intensity in [1, 2, 3]:
        y = []
        for pert_type in perturbation_types:
            value = df[(df['config.typo_type'] == pert_type) & 
                       (df['config.typo_intensity'] == intensity) & 
                       (df['config.model_name'] == model)][f'result.{metric}'].iloc[0]
            y.append(value)
        
        fig.add_trace(go.Bar(
            x=perturbation_types,
            y=y,
            name=f'Intensity {intensity}',
            marker_color=intensity_colors[intensity]
        ))
    
    fig.add_shape(
        type="line",
        x0=-0.5,
        x1=len(perturbation_types)-0.5,
        y0=baseline_value,
        y1=baseline_value,
        line=dict(color="black", width=2, dash="dash")
    )
    
    num_bits = get_model_num_bits(model)
    fig.update_layout(
        title=f'All Perturbations Bar Plot of {metric} for {model} ({num_bits}-bit)',
        xaxis_title="Perturbation Type",
        yaxis_title=metric,
        barmode='group',
        height=600,
        width=1200
    )
    fig.update_xaxes(tickangle=45)
    
    return fig

def create_grouped_bar_plots(df, metric, intensity):
    groups = [
        ['char_insertion', 'char_deletion', 'char_replacement', 'char_repetition'],
        ['char_swapping', 'char_LCC', 'char_insert_noise', 'char_substitution'],
        ['word_CMW', 'word_remove_punctuation', 'word_internet_slang', 'word_emoji'],
        ['word_synonym', 'word_phrase_translation', 'word_context_aware_insertion', 'word_keyword_only'],
        ['word_taxonomy_neg', 'word_taxonomy_pos', 'word_repeat']
    ]
    
    fig = make_subplots(rows=3, cols=2, subplot_titles=[f"Group {i+1}" for i in range(5)],
                        vertical_spacing=0.1, horizontal_spacing=0.05,
                        specs=[[{"type": "bar"}]*2]*3)
    
    df_intensity = df[df['config.typo_intensity'] == intensity]
    
    for i, group in enumerate(groups):
        row = i // 2 + 1
        col = i % 2 + 1
        
        for pert_type in group:
            for model in model_colors.keys():
                value = df_intensity[(df_intensity['config.typo_type'] == pert_type) & 
                                     (df_intensity['config.model_name'] == model)][f'result.{metric}'].iloc[0]
                
                fig.add_trace(
                    go.Bar(
                        x=[f"{pert_type}_{model}"],
                        y=[value],
                        name=model,
                        marker_color=model_colors[model],
                        showlegend=i==0  # Only show legend for the first group
                    ),
                    row=row, col=col
                )
    
    fig.update_layout(
        title=f'Models Comparison Bar Plot of {metric} for Intensity {intensity}',
        height=1500,
        width=1800,
        barmode='group',
        bargap=0.15,
        bargroupgap=0.1
    )
    
    # Update x-axis for each subplot
    for i in range(1, 6):
        fig.update_xaxes(tickangle=45, title_text="Perturbation Type", row=(i-1)//2+1, col=(i-1)%2+1)
        fig.update_yaxes(title_text=metric if i % 2 else "", row=(i-1)//2+1, col=(i-1)%2+1)
    
    return fig


def generate_pdf_report(plots, pdf_path, all_results):
    pdf = FPDF()
    pdf.set_auto_page_break(auto=True, margin=15)
    
    # Add title page with configuration details
    pdf.add_page()
    pdf.set_font("Arial", 'B', size=16)
    pdf.cell(0, 10, "Model Comparison: Typo Effect Analysis", ln=True, align='C')
    pdf.set_font("Arial", size=12)
    
    # Print configuration details
    pdf.cell(0, 10, "Configuration Details:", ln=True)
    config_columns = [col for col in all_results.columns if col.startswith('config.')]
    for col in config_columns:
        unique_values = all_results[col].unique()
        pdf.cell(0, 10, f"{col}: {', '.join(map(str, unique_values))}", ln=True)

    # Add plots to the PDF
    for plot_file, desc in plots:
        pdf.add_page()
        pdf.set_font("Arial", 'B', size=14)
        pdf.cell(0, 20, desc, ln=True, align='C')
        
        with Image.open(plot_file) as img:
            img_width, img_height = img.size
        
        scale_factor = (pdf.w - 20) / img_width
        scaled_height = img_height * scale_factor
        
        pdf.image(plot_file, x=10, y=pdf.get_y(), w=pdf.w-20, h=scaled_height)

    pdf.output(pdf_path, "F")

def main(all_results, exp_id):
    plots_dir = f"plots/model_comparison_{exp_id}"
    os.makedirs(plots_dir, exist_ok=True)

    plots = []
    metrics = ['Accuracy', 'AUCPR_sample']

    for metric in metrics:
        # Radar plots for each intensity
        for intensity in [1, 2, 3]:
            fig = create_comparison_radar_plots(all_results, metric, intensity)
            plot_file = os.path.join(plots_dir, f"{metric}_Radar_Plot_Intensity_{intensity}_{exp_id}.png")
            fig.write_image(plot_file, scale=2)
            plots.append((plot_file, f"Comparison Radar Plot of {metric} for Intensity {intensity}"))
        
        # Boxplot Comparison
        fig = create_boxplot_comparison(all_results, metric)
        plot_file = os.path.join(plots_dir, f"{metric}_Boxplot_Comparison_{exp_id}.png")
        fig.write_image(plot_file, scale=2)
        plots.append((plot_file, f"Boxplot Comparison of {metric}"))
        
        # All Perturbations Bar Plot for each model
        for model in model_colors.keys():
            fig = create_all_perturbations_bar_plot(all_results, metric, model)
            plot_file = os.path.join(plots_dir, f"{metric}_All_Perturbations_Bar_Plot_{model}_{exp_id}.png")
            fig.write_image(plot_file, scale=2)
            plots.append((plot_file, f"All Perturbations Bar Plot of {metric} for {model}"))

    # Generate the PDF report
    pdf_path = os.path.join(plots_dir, f"model_comparison_report_{exp_id}.pdf")
    generate_pdf_report(plots, pdf_path, all_results)

    print(f"PDF report generated and saved as '{pdf_path}'")

if __name__ == "__main__":
    # Load your results dataframe here
    # Assuming you've already loaded the all_results dataframe
    
    # Specify the experiment ID
    exp_id = "awq_hqq_bnb_comparison-10-19"
    
    # Call the main function with the experiment ID
    main(all_results, exp_id)

KeyboardInterrupt: 

In [10]:
all_results["config.model_name"].unique()

array(['Llama-3-8B', 'Llama-3-8B-AWQ-4bit-local',
       'Llama-3-8B-BNB-4bit-local', 'Llama-3-8B-HQQ-mixed-local'],
      dtype=object)

## 4. Remove Duplicates

In [5]:
import pandas as pd

def inspect_and_remove_duplicates(df):
    # Define the columns used for pivoting
    pivot_columns = ['config.typo_type', 'config.typo_intensity']
    
    # Find duplicates in pivot columns
    duplicate_mask = df.duplicated(subset=pivot_columns, keep=False)
    duplicates = df[duplicate_mask]
    
    if duplicates.empty:
        print("No duplicates found in pivot columns.")
        return df
    
    print("Duplicate entries found in pivot columns:")
    print(duplicates[pivot_columns])
    
    print("\nFull rows for duplicate entries:")
    print(duplicates)
    
    # Ask user how to handle duplicates
    print("\nHow would you like to handle these duplicates?")
    print("1: Keep first occurrence")
    print("2: Keep last occurrence")
    print("3: Remove all duplicates")
    print("4: Do nothing (keep all)")
    
    choice = input("Enter your choice (1-4): ")
    
    if choice == '1':
        df_cleaned = df.drop_duplicates(subset=pivot_columns, keep='first')
        print(f"Removed {len(df) - len(df_cleaned)} duplicate rows.")
    elif choice == '2':
        df_cleaned = df.drop_duplicates(subset=pivot_columns, keep='last')
        print(f"Removed {len(df) - len(df_cleaned)} duplicate rows.")
    elif choice == '3':
        df_cleaned = df.drop_duplicates(subset=pivot_columns, keep=False)
        print(f"Removed {len(df) - len(df_cleaned)} duplicate rows.")
    elif choice == '4':
        df_cleaned = df
        print("No rows removed.")
    else:
        print("Invalid choice. No rows removed.")
        df_cleaned = df
    
    return df_cleaned

# Assuming your dataframe is named 'all_results'
all_results_llama = inspect_and_remove_duplicates(all_results_llama)

# You can now use all_results_cleaned for further processing

Duplicate entries found in pivot columns:
           config.typo_type  config.typo_intensity
42  word_phrase_translation                      1
43  word_phrase_translation                      2
59  word_phrase_translation                      1
60  word_phrase_translation                      2

Full rows for duplicate entries:
    _id  config.overwrite config.db_collection config.dataset_name  \
42   43                43      llama-typo-eval                 P17   
43   44                44      llama-typo-eval                 P17   
59   61                61      llama-typo-eval                 P17   
60   62                62      llama-typo-eval                 P17   

   config.device    config.exp_id config.max_entries  config.max_new_tokens  \
42          cuda  typo-test-10-01               None                     25   
43          cuda  typo-test-10-01               None                     25   
59          cuda  typo-test-10-01               None                     25   
60